# MyCON — General-purpose Reconstruction / Round 4
Run on Colab using your own video path(s), with a new run for each scene. A second video is a held-out capture of the SAME scene, not a second training clip. Single-clip results are exploratory temporal holdout. No historical default, no GS training, no BIM or metric claims. Select GPU runtime for Dense MVS. Source video is read-only; outputs go to a new project/run directory.

In [ ]:
#@title 1. Project configuration (edit before running)
PROJECT = "my_project" #@param {type:"string"}
TRAIN_VIDEO = "" #@param {type:"string"}
TEST_VIDEO = "" #@param {type:"string"}
TRAIN_SESSION = "capture_train" #@param {type:"string"}
TEST_SESSION = "capture_test" #@param {type:"string"}
EXPECTED_TRAIN_SHA256 = "" #@param {type:"string"}
EXPECTED_TEST_SHA256 = "" #@param {type:"string"}
SAMPLE_HZ = 2.0 #@param {type:"number"}
MAX_FRAMES = 120 #@param {type:"integer"}
MIN_BLUR = 35.0 #@param {type:"number"}
EMBARGO_SEC = 1.0 #@param {type:"number"}
MAX_DENSE_SIZE = 1600 #@param {type:"integer"}
INSTALL_CPU_COLMAP = True #@param {type:"boolean"}
DENSE_MVS = True #@param {type:"boolean"}
DENSE_BACKEND = "pycolmap_cuda12" #@param ["pycolmap_cuda12","cli_cuda"]
COLMAP_EXECUTABLE = "colmap" #@param {type:"string"}
APPROVED_CUDA_BINARY_SHA256 = "" #@param {type:"string"}
# This notebook deliberately contains no Gaussian trainer.
HEAVY_GS_TRAINING = False
CODE_SHA = "efe2e8aaf1af8866cad8f36ea858e6941fbf4e3c"
import os, re, sys, json, shutil, hashlib, subprocess, uuid, datetime, traceback
from pathlib import Path
assert re.fullmatch(r"[A-Za-z0-9_-]{2,64}", PROJECT)
assert re.fullmatch(r"[A-Za-z0-9_-]{2,70}", TRAIN_SESSION)
assert not TEST_VIDEO or (re.fullmatch(r"[A-Za-z0-9_-]{2,70}", TEST_SESSION) and TEST_SESSION != TRAIN_SESSION)
assert TRAIN_VIDEO, "Set a real source path before proceeding"
assert 0 < SAMPLE_HZ <= 15 and 10 <= MAX_FRAMES <= 180 and 320 <= MAX_DENSE_SIZE <= 4096
assert MIN_BLUR >= 0 and EMBARGO_SEC >= 0
assert not HEAVY_GS_TRAINING
RUN_ID = "recon-"+datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:8]
WORK = Path("/content/MyCON_R4_Isolated") / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
RUN_ROOT = WORK/"runs"
RUN_ROOT.mkdir()
RUN_DIR = RUN_ROOT/RUN_ID
INSTALL_LOG = WORK/"environment.log"
print(RUN_ID)

In [ ]:
#@title 2. Mount Drive, verify paths and pin code
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/MyCON_Reconstruction_General_Runs")/PROJECT/RUN_ID
DRIVE_ROOT.mkdir(parents=True, exist_ok=False)
def digest(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""): h.update(chunk)
    return h.hexdigest()
def logcmd(args, check=True):
    with INSTALL_LOG.open("a",encoding="utf-8") as f:
        f.write("\n"+json.dumps(args)+"\n"); f.flush()
        p=subprocess.run(args,stdout=f,stderr=subprocess.STDOUT,check=False)
        f.write("exit="+str(p.returncode)+"\n")
    if check and p.returncode: raise RuntimeError("Command failed: "+str(args))
    return p.returncode
def snapshot(stage, source):
    dest=DRIVE_ROOT/stage
    dest.mkdir(exist_ok=False)
    target=dest/source.name
    shutil.copytree(source,target)
    receipt={}
    for p in source.rglob("*"):
        if p.is_file():
            assert not p.is_symlink()
            r=p.relative_to(source)
            assert digest(p)==digest(target/r), str(r)
            receipt[r.as_posix()]={"sha256":digest(target/r),"bytes":p.stat().st_size}
    with (dest/"snapshot_receipt.json").open("x") as f:
        json.dump({"run_id":RUN_ID,"stage":stage,"files":receipt},f,indent=2)
    return target
paths=[Path(TRAIN_VIDEO).expanduser().resolve()]
if TEST_VIDEO: paths.append(Path(TEST_VIDEO).expanduser().resolve())
assert len(set(paths))==len(paths)
assert all(p.is_file() and not p.is_relative_to(DRIVE_ROOT.resolve()) for p in paths)
REPO=WORK/"code"
logcmd(["git","init",str(REPO)])
logcmd(["git","-C",str(REPO),"remote","add","origin","https://github.com/sayyedalimrj/MyCon.git"])
logcmd(["git","-C",str(REPO),"fetch","--depth=1","origin",CODE_SHA])
logcmd(["git","-C",str(REPO),"checkout","--detach","FETCH_HEAD"])
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==CODE_SHA
os.chdir(REPO/"repository")
sys.path.insert(0,str(Path.cwd()))
print("Pinned:",CODE_SHA)

### Installation notes
Ubuntu repository COLMAP normally has no CUDA. It is suitable for the CPU sparse mapping path. The default dense path installs the official pinned CUDA12 PyCOLMAP wheel separately; a user-supplied CUDA CLI is allowed only with an expected binary SHA. A successful import or GPU listing is not proof of MVS success. The actual command exit and nonempty output are required. See https://colmap.github.io/install.html and https://pypi.org/project/pycolmap-cuda12/3.13.0/.

In [ ]:
#@title 3. Controlled dependencies and preflight
from importlib.util import find_spec
install_state={}
try:
    if INSTALL_CPU_COLMAP and not shutil.which(COLMAP_EXECUTABLE):
        logcmd(["apt-get","update"])
        logcmd(["apt-get","install","-y","colmap"])
    if not find_spec("cv2") or not find_spec("numpy") or not find_spec("skimage"):
        logcmd([sys.executable,"-m","pip","install","-r",str(Path.cwd()/"requirements-research-reconstruction.txt")])
    executable=shutil.which(COLMAP_EXECUTABLE)
    if not executable: raise RuntimeError("COLMAP binary unavailable")
    executable=str(Path(executable).resolve())
    logcmd([executable,"-h"])
    if DENSE_MVS and DENSE_BACKEND=="cli_cuda":
        assert APPROVED_CUDA_BINARY_SHA256 and digest(executable)==APPROVED_CUDA_BINARY_SHA256
    install_state["sparse"]="available"
except Exception as exc:
    install_state["sparse"]="blocked: "+str(exc)
    executable=None
if DENSE_MVS and DENSE_BACKEND=="pycolmap_cuda12":
    try:
        logcmd(["nvidia-smi","-L"])
        logcmd([sys.executable,"-m","pip","install","pycolmap-cuda12==3.13.0"])
        logcmd([sys.executable,"-c","import pycolmap as p; print(p.__version__,p.has_cuda); assert p.__version__=='3.13.0' and p.has_cuda"])
        install_state["dense"]="wheel_preflight_only"
    except Exception as exc:
        install_state["dense"]="blocked: "+str(exc)
elif DENSE_MVS:
    install_state["dense"]="cli_cuda_selected_not_executed"
else: install_state["dense"]="not_requested"
from reconstruction.preflight import audit
preflight=audit()
preflight.update(code_sha=CODE_SHA,install_state=install_state,colmap_binary_sha256=digest(executable) if executable else None)
logcmd([sys.executable,"-m","pip","freeze"],check=False)
with (WORK/"preflight.json").open("x") as f: json.dump(preflight,f,indent=2)
shutil.copy2(INSTALL_LOG,DRIVE_ROOT/"environment.log")
shutil.copy2(WORK/"preflight.json",DRIVE_ROOT/"preflight.json")
print(json.dumps(preflight,indent=2))

In [ ]:
#@title 4. Read-only source hash and verified local copy
from reconstruction.workflow import prepare
import cv2, math
CACHE=WORK/"inputs"
CACHE.mkdir()
source_receipts=[]
copies=[]
for i,p in enumerate(paths):
    expected=[EXPECTED_TRAIN_SHA256,EXPECTED_TEST_SHA256][i].strip().lower()
    h=digest(p)
    if expected: assert expected==h, "Source hash mismatch"
    local=CACHE/("source_"+str(i)+p.suffix.lower())
    with p.open("rb") as f,local.open("xb") as g: shutil.copyfileobj(f,g,8*1024*1024)
    assert digest(local)==h and digest(p)==h
    c=cv2.VideoCapture(str(local))
    fps=float(c.get(cv2.CAP_PROP_FPS)); count=int(c.get(cv2.CAP_PROP_FRAME_COUNT)); c.release()
    assert math.isfinite(fps) and fps>0 and count>=2
    source_receipts.append({"original_name":p.name,"sha256":h,"size":p.stat().st_size,"expected_hash_supplied":bool(expected),"fps":fps,"frames":count,"duration_s":count/fps})
    copies.append(local)
assert len({r["sha256"] for r in source_receipts})==len(source_receipts)
# Avoid excessive RAM and cap decoded candidates; retain coverage over the full duration.
effective_hz=min(SAMPLE_HZ, min(min(160,MAX_FRAMES*10)/r["duration_s"] for r in source_receipts))
print("Effective sampling Hz",effective_hz)
m=prepare(run_root=RUN_ROOT,run_id=RUN_ID,train_video=copies[0],train_session=TRAIN_SESSION,
          test_video=copies[1] if len(copies)>1 else None,test_session=TEST_SESSION if len(copies)>1 else None,
          sample_hz=effective_hz,max_frames=MAX_FRAMES,min_blur=MIN_BLUR,min_mean=25,max_mean=230,max_clip=.38,embargo_sec=EMBARGO_SEC)
with (RUN_DIR/"input_receipt.json").open("x") as f: json.dump({"sources":source_receipts,"code_sha":CODE_SHA,"effective_hz":effective_hz},f,indent=2)
for r in m["source_videos"].values():
    assert r["frames_decoded"] >= r["frames_reported"]-2, "Decoder stopped early; do not claim full video coverage"
print(m["counts"],m["train_test_protocol"])
selection_snapshot=snapshot("01_selection",RUN_DIR)
print("Saved",selection_snapshot)

In [ ]:
#@title 5. Real fresh COLMAP (train mapper then test localization)
from reconstruction.workflow import run_colmap, train_plan
from reconstruction.coverage import coverage
sfm={"status":"not_executed"}
try:
    if executable is None: raise RuntimeError("Missing COLMAP executable, inspect environment.log")
    sfm=run_colmap(RUN_DIR,executable,execute=True)
except Exception as exc:
    sfm={"status":"blocked_or_failed","reason":str(exc)}
    with (RUN_DIR/"sfm_error.log").open("x") as f: f.write(traceback.format_exc())
if not (RUN_DIR/"coverage_report.json").exists(): coverage(RUN_DIR)
with (RUN_DIR/"sfm_status.json").open("x") as f: json.dump(sfm,f,indent=2)
print(json.dumps(sfm,indent=2))
# Partial failures are archived, never resumed in-place.
sfm_snapshot=snapshot("02_sfm",RUN_DIR)
print("Saved",sfm_snapshot)

In [ ]:
#@title 6. Sparse export, undistortion, train-only GS staging and Dense MVS
from reconstruction.general_pilot import stage
products={"status":"not_executed_no_fresh_sfm","gs_training":"not_executed"}
if sfm.get("status")=="colmap_executed":
    try:
        products=stage(RUN_DIR,executable,max_image_size=MAX_DENSE_SIZE,run_dense=DENSE_MVS,dense_backend=DENSE_BACKEND)
    except Exception as exc:
        products={"status":"blocked_or_failed","reason":str(exc),"gs_training":"not_executed"}
        with (RUN_DIR/"products_error.log").open("x") as f: f.write(traceback.format_exc())
with (RUN_DIR/"products_status.json").open("x") as f: json.dump(products,f,indent=2)
if not (RUN_DIR/"gs_training_plan.json").exists(): train_plan(RUN_DIR)
assert not HEAVY_GS_TRAINING
print(json.dumps({k:v for k,v in products.items() if k!="files"},indent=2))

In [ ]:
#@title 7. Final immutable snapshot and A/V handoff
from reconstruction.workflow import receipt
for p,r in zip(paths,source_receipts):
    assert digest(p)==r["sha256"], "Original changed externally during run"
with (RUN_DIR/"completion_status.json").open("x") as f:
    json.dump({"run_id":RUN_ID,"source_unchanged":True,"fresh_colmap":sfm.get("status")=="colmap_executed",
               "same_run_products":products.get("schema_version")=="mycon.reconstruction.same_run_handoff.v1",
               "dense_status":products.get("dense_status","not_executed"),"gs_training":"not_executed",
               "metric_scale_verified":False,"field_validated":False},f,indent=2)
receipt(RUN_DIR)
FINAL=snapshot("03_final",RUN_DIR)
print("Final run:",FINAL)
print("For V: dataset_manifest.json and camera_poses.json + original images/ (registered views only).")
print("For A: same_run_products/av_handoff.json + sparse_train.ply, dense_train_fused.ply only if verified present.")
print("No historical cloud may be overlaid without an independently verified transform.")
print("GS: staged only; NO TRAINING. Checkpoint recovery from prior runs unchanged.")

## Recovery / review
Each snapshot is create-only and hash verified. If Colab disconnects, keep the last completed snapshot; run again with a new ID rather than overwrite any outputs. The final snapshot can be large because it includes MVS working data. A nonempty cloud is an unscaled candidate, not an accepted wall or survey. Viewer team may invoke its existing `producer_bridge` with the final run directory; do not edit V code here. Historical GS6999 / old R4 files are never used as defaults or training inputs. Heavy GS training must be separately authorized; this notebook has no trainer.